# Conversational AI

In [1]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from pypdf import PdfReader

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [3]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4o-mini'

In [4]:
system_message = """You are a public funding expert analyzing grant programmes. 
You are provided with guideline documents of grant programmes and extract important information.

Always follow these rules:
- If you cannot find an information asked for, say so and do not make something up.
- If the material provided covers multiple funding programmes, provide information always specifically for each programme.

If the guideline documents provide this information state:
- Size of possible funding
- Type of possible funding (grant or loan)
- Project duration
- Contact person
"""

In [5]:
# extract text from pdf
def pdf_extractor(pdf_path):
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PdfReader(file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text() + "\n"
            return text.strip()
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

In [13]:
def extract_pdf_text(pdf_path):
    text = ""
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PdfReader(file)
            for page_num, page in enumerate(pdf_reader.pages, start=1):
                page_text = page.extract_text()
                text += f"--- Seite {page_num} ---\n{page_text}\n"
            return text.strip()
    except FileNotFoundError:
        return f"File '{pdf_path}' not found."
    except Exception as e:
        return f"Error reading PDF: {str(e)}"

In [14]:
pdf_text = extract_pdf_text("Basisprogramm+Ausschreibungsleitfaden_Deutsch.pdf")
print(pdf_text)

--- Seite 1 ---
 
 
  
EINREICHUNG JEDERZEIT  MÖGLICH  
 
VERSION 5.2 
GÜLTIG AB 1. SEPTEMBER 2025  
 
– 
AUSSCHREIBUNGSLEITFADEN FÜR 
BASISPROGRAMM  
--- Seite 2 ---
 
 
© Österreichische  Basisprogramm  Ausschreibungsleitfaden  Version 5.2 
Forschungsför derungsgesellschaft  gültig ab 1. September 2025 Seite 2/9 
INHALTSVERZEICHNIS  
1 DAS WICHTIGSTE IN KÜRZE  ................................ ..................  3 
2 ZIELE DER AUSSCHREIBUNG  ................................ .................  5 
3 SCHWERPUNKTE DER AUSSCHREIBUNG  ................................  6 
4 AUSSCHREIBUNGSDOKUMENTE  ................................ ...........  7 
5 FÖRDERUNGSENTSCHEIDUNG UND RECHTSGRUNDLAGEN  .... 8 
6 WEITERE INFORMATIONEN  ................................ ..................  8 
6.1 Unterstützung der Öffentlichkeitsarbeit  ................................ ............  8 
6.2 Service FFG Projektdatenbank  ................................ ...........................  9 
6.3 Weitere Förderun

In [7]:
# chat generator function for gradio
def chat(message, history, pdf_path=None, system_message=system_message):
    pdf_text = None
    if pdf_path:
        pdf_text = pdf_extractor(pdf_path)
        system_message += f"\nThis is the pdf:\n{pdf_text}"

    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [8]:
gr.ChatInterface(fn=chat, additional_inputs=[gr.File(label="upload pdf")], type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
